In [1]:
# Standard library imports
import os
import sys
import csv
import time
import random
from pathlib import Path
from datetime import datetime
from typing import Dict, Optional, Tuple, List
from collections import defaultdict

# Scientific computing
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
# Note: If this hangs, try: pip install --upgrade torch torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
from tqdm import tqdm

print(f"✓ PyTorch {torch.__version__} loaded")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
print("✓ All imports complete")

✓ PyTorch 2.9.1 loaded
✓ CUDA available: False
✓ All imports complete


In [2]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================
# Change these values to customize training
# ============================================================================

# Paths (uses same portable BASE_DIR as preprocessing notebook)
BASE_DIR = Path(os.getenv("PIXELCLEAR_ROOT", Path.cwd()))
INDEX_DIR = BASE_DIR / "indices"
OUTPUT_DIR = BASE_DIR / "experiments" / "baseline_training"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify data indices exist
print("\n" + "="*60)
print("Data Path Verification")
print("="*60)
train_csv = INDEX_DIR / "gopro_train.csv"
val_csv = INDEX_DIR / "gopro_test.csv"

if not train_csv.exists():
    print(f"⚠️  WARNING: {train_csv} not found!")
    print("   Run 01_data_preprocessing.ipynb first to create indices.")
    raise FileNotFoundError(f"Training CSV not found: {train_csv}")
else:
    print(f"✓ Train CSV found: {train_csv}")

if not val_csv.exists():
    print(f"⚠️  WARNING: {val_csv} not found!")
    print("   Run 01_data_preprocessing.ipynb first to create indices.")
    raise FileNotFoundError(f"Validation CSV not found: {val_csv}")
else:
    print(f"✓ Val CSV found: {val_csv}")
print("="*60)

# Model configuration
MODEL_CONFIG = {
    "base_channels": 32,      # Model width (16=tiny, 32=base, 48=large)
    "num_blocks": 4,          # Number of NAF blocks
    "dropout": 0.0,           # Dropout rate (0.0 for baseline)
}

# Training configuration
TRAIN_CONFIG = {
    "batch_size": 8,          # Batch size
    "patch_size": 256,        # Input patch size
    "epochs": 5,              # Number of epochs (set to 5 for quick testing)
    "learning_rate": 1e-4,    # Initial learning rate
    "weight_decay": 1e-4,      # Weight decay for AdamW
    "scheduler": "cosine",    # "cosine" or "step"
    "scheduler_params": {
        "T_max": 5,           # For cosine: max epochs (updated to match epochs)
        "eta_min": 1e-6,      # For cosine: min LR
        "step_size": 20,      # For step: decay every N epochs
        "gamma": 0.5,         # For step: LR decay factor
    },
    "grad_clip": 1.0,          # Gradient clipping (0.0 to disable)
    "use_amp": True,          # Mixed precision training (if CUDA)
    "num_workers": 0,         # DataLoader workers (0 for notebook compatibility)
}

# Loss configuration
LOSS_CONFIG = {
    "l1_weight": 1.0,          # L1 loss weight
    "ssim_weight": 0.1,        # SSIM loss weight (0.0 to disable)
    "use_charbonnier": False,  # Use Charbonnier loss instead of L1
    "charbonnier_eps": 1e-3,  # Epsilon for Charbonnier
}

# Dataset configuration
DATASET_CONFIG = {
    "train_csv": str(INDEX_DIR / "gopro_train.csv"),
    "val_csv": str(INDEX_DIR / "gopro_test.csv"),
    "adaptive_sampling": True,  # Use degradation-aware patch sampling
    "num_candidates": 5,        # Patch candidates for adaptive sampling
}

# Multi-dataset configuration (for later use)
MULTI_DATASET_CONFIG = {
    "enabled": False,  # Set to True after baseline works
    "weights": {
        "gopro": 0.6,
        "realblur": 0.3,
        "sidd": 0.1,
    },
    "csvs": {
        "gopro": str(INDEX_DIR / "gopro_train.csv"),
        "realblur": str(INDEX_DIR / "realblur_j_train.csv"),
        "sidd": str(INDEX_DIR / "sidd_train.csv"),
    },
}

# Sanity overfit test configuration
SANITY_CONFIG = {
    "enabled": True,           # Run sanity test first
    "num_samples": 100,       # Use only N samples
    "epochs": 5,               # Train for few epochs
    "expected_loss_drop": 0.3, # Loss should drop by this amount
}

# Validation and logging
VAL_CONFIG = {
    "val_interval": 5,         # Validate every N epochs (changed from 1 to speed up training)
    "save_interval": 5,        # Save checkpoint every N epochs
    "save_images": True,       # Save image grids
    "num_val_images": 4,      # Number of images to visualize
    "max_val_samples": 100,   # Limit validation set size for faster validation
    "metrics_csv": str(OUTPUT_DIR / "metrics.csv"),
}

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Resume training configuration
RESUME_CONFIG = {
    "enabled": False,          # Set to True to resume from checkpoint
    "checkpoint_path": str(OUTPUT_DIR / "checkpoints" / "best_model.pth"),
    "start_epoch": 0,          # Will be set from checkpoint if resuming
}

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n{'='*60}")
print(f"Training Configuration")
print(f"{'='*60}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("⚠️  WARNING: Training on CPU will be slow!")
    print("   Consider using GPU (Colab, Kaggle, or local GPU)")
    print("   For faster training, set CUDA_VISIBLE_DEVICES or use cloud GPU")
else:
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"Epochs: {TRAIN_CONFIG['epochs']}")
print(f"Learning rate: {TRAIN_CONFIG['learning_rate']}")
print(f"Validation interval: Every {VAL_CONFIG['val_interval']} epochs")
print(f"Sanity test: {SANITY_CONFIG['enabled']}")
if RESUME_CONFIG["enabled"]:
    print(f"Resume training: Yes (from {RESUME_CONFIG['checkpoint_path']})")
print(f"{'='*60}\n")


Data Path Verification
✓ Train CSV found: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear/indices/gopro_train.csv
✓ Val CSV found: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear/indices/gopro_test.csv

Training Configuration
Device: cpu
⚠️  WARNING: Training on CPU will be slow!
   Consider using GPU (Colab, Kaggle, or local GPU)
   For faster training, set CUDA_VISIBLE_DEVICES or use cloud GPU
Output directory: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear/experiments/baseline_training
Batch size: 8
Epochs: 5
Learning rate: 0.0001
Validation interval: Every 5 epochs
Sanity test: True



In [3]:
# ============================================================================
# LIGHTWEIGHT BASELINE RESTORATION MODEL
# ============================================================================
# NAFNet-inspired architecture: Simple, fast, effective
# ============================================================================

class SimpleGate(nn.Module):
    """Simple gating mechanism for activation."""
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class NAFBlock(nn.Module):
    """NAFNet block: Simple and effective restoration block."""
    def __init__(self, channels: int, dropout: float = 0.0):
        super().__init__()
        self.dwconv = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels)
        self.norm = nn.LayerNorm(channels)
        self.pwconv1 = nn.Conv2d(channels, channels * 2, kernel_size=1)
        self.act = SimpleGate()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.pwconv2 = nn.Conv2d(channels, channels, kernel_size=1)
        
    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)  # [B, C, H, W] -> [B, H, W, C]
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2)  # [B, H, W, C] -> [B, C, H, W]
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.pwconv2(x)
        return x + residual


class BaselineRestorationModel(nn.Module):
    """
    Lightweight baseline restoration model.
    
    Architecture:
    - Input conv: 3 -> base_channels
    - N NAFBlocks for feature extraction
    - Output conv: base_channels -> 3
    - Residual connection from input
    """
    def __init__(self, base_channels: int = 32, num_blocks: int = 4, dropout: float = 0.0):
        super().__init__()
        
        self.input_conv = nn.Conv2d(3, base_channels, kernel_size=3, padding=1)
        
        # Stack of NAF blocks
        self.blocks = nn.Sequential(*[
            NAFBlock(base_channels, dropout=dropout)
            for _ in range(num_blocks)
        ])
        
        self.output_conv = nn.Conv2d(base_channels, 3, kernel_size=3, padding=1)
        
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input degraded image [B, 3, H, W]
            
        Returns:
            Restored image [B, 3, H, W]
        """
        residual = x
        x = self.input_conv(x)
        x = self.blocks(x)
        x = self.output_conv(x)
        return x + residual  # Residual connection


# Test model creation
def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Create model
model = BaselineRestorationModel(
    base_channels=MODEL_CONFIG["base_channels"],
    num_blocks=MODEL_CONFIG["num_blocks"],
    dropout=MODEL_CONFIG["dropout"]
).to(DEVICE)

num_params = count_parameters(model)
print(f"\n{'='*60}")
print(f"Model: BaselineRestorationModel")
print(f"{'='*60}")
print(f"Parameters: {num_params:,} ({num_params/1e6:.2f}M)")
print(f"Base channels: {MODEL_CONFIG['base_channels']}")
print(f"Number of blocks: {MODEL_CONFIG['num_blocks']}")

# Test forward pass
dummy_input = torch.randn(1, 3, 256, 256).to(DEVICE)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")
print(f"✓ Model forward pass successful!")
print(f"{'='*60}\n")


Model: BaselineRestorationModel
Parameters: 15,971 (0.02M)
Base channels: 32
Number of blocks: 4
Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
✓ Model forward pass successful!



In [4]:
# ============================================================================
# DATA PREPROCESSING COMPONENTS (Self-Contained)
# ============================================================================
# All data loading functions included here - no need to run preprocessing notebook
# ============================================================================

class DegradationAwarePatchSampler:
    """
    Selects patches based on multiple degradation types.
    """
    
    def __init__(
        self, 
        patch_size: int = 256, 
        num_candidates: int = 5,
        mode: str = "train",
        dataset_name: str = "GoPro",
        min_edge_density: float = 0.01
    ):
        self.patch_size = patch_size
        self.num_candidates = num_candidates
        self.mode = mode
        self.min_edge_density = min_edge_density
        self.weights = self._get_dataset_weights(dataset_name)
    
    def _get_dataset_weights(self, dataset_name: str) -> Dict[str, float]:
        if "GoPro" in dataset_name:
            return {"blur": 0.5, "noise": 0.2, "lowlight": 0.15, "compression": 0.15}
        elif "SIDD" in dataset_name:
            return {"blur": 0.2, "noise": 0.5, "lowlight": 0.15, "compression": 0.15}
        elif "RealBlur" in dataset_name:
            return {"blur": 0.35, "noise": 0.25, "lowlight": 0.25, "compression": 0.15}
        else:
            return {"blur": 0.3, "noise": 0.3, "lowlight": 0.2, "compression": 0.2}
    
    def compute_blur_score(self, img_tensor: torch.Tensor) -> float:
        if img_tensor.dim() == 3:
            gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
        else:
            gray = img_tensor
        laplacian_kernel = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32).view(1, 1, 3, 3)
        gray_4d = gray.unsqueeze(0).unsqueeze(0)
        laplacian = F.conv2d(gray_4d, laplacian_kernel, padding=1)
        variance = torch.var(laplacian).item()
        return -variance
    
    def compute_noise_score(self, img_tensor: torch.Tensor) -> float:
        if img_tensor.dim() == 3:
            gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
        else:
            gray = img_tensor
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        gray_4d = gray.unsqueeze(0).unsqueeze(0)
        grad_x = F.conv2d(gray_4d, sobel_x, padding=1)
        grad_y = F.conv2d(gray_4d, sobel_y, padding=1)
        gradient_mag = torch.sqrt(grad_x**2 + grad_y**2 + 1e-8)
        flat_mask = gradient_mag < 0.1
        if flat_mask.sum() > 0:
            flat_pixels = gray_4d[flat_mask]
            noise_score = torch.var(flat_pixels).item()
        else:
            noise_score = torch.var(gray).item()
        return noise_score
    
    def compute_lowlight_score(self, img_tensor: torch.Tensor) -> float:
        if img_tensor.dim() == 3:
            gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
        else:
            gray = img_tensor
        mean_brightness = torch.mean(gray).item()
        brightness_score = 1.0 - mean_brightness
        contrast = torch.std(gray).item()
        contrast_score = 1.0 / (contrast + 0.1)
        return brightness_score * 0.6 + contrast_score * 0.4
    
    def compute_compression_score(self, img_tensor: torch.Tensor) -> float:
        if img_tensor.dim() == 3:
            gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
        else:
            gray = img_tensor
        h, w = gray.shape
        block_size = 8
        vertical_edges = 0
        for i in range(block_size, h, block_size):
            if i < h - 1:
                diff = torch.abs(gray[i, :] - gray[i-1, :])
                vertical_edges += torch.mean(diff).item()
        horizontal_edges = 0
        for j in range(block_size, w, block_size):
            if j < w - 1:
                diff = torch.abs(gray[:, j] - gray[:, j-1])
                horizontal_edges += torch.mean(diff).item()
        num_v_edges = (h // block_size) - 1
        num_h_edges = (w // block_size) - 1
        if num_v_edges > 0:
            vertical_edges /= num_v_edges
        if num_h_edges > 0:
            horizontal_edges /= num_h_edges
        compression_score = (vertical_edges + horizontal_edges) / 2.0
        return compression_score
    
    def compute_edge_density(self, img_tensor: torch.Tensor) -> float:
        if img_tensor.dim() == 3:
            gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
        else:
            gray = img_tensor
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        gray_4d = gray.unsqueeze(0).unsqueeze(0)
        grad_x = F.conv2d(gray_4d, sobel_x, padding=1)
        grad_y = F.conv2d(gray_4d, sobel_y, padding=1)
        gradient_mag = torch.sqrt(grad_x**2 + grad_y**2 + 1e-8)
        edge_threshold = 0.1
        edge_pixels = (gradient_mag > edge_threshold).sum().item()
        total_pixels = gradient_mag.numel()
        edge_density = edge_pixels / total_pixels
        return edge_density
    
    def compute_degradation_score(self, img_tensor: torch.Tensor) -> float:
        blur_raw = self.compute_blur_score(img_tensor)
        noise_raw = self.compute_noise_score(img_tensor)
        lowlight_raw = self.compute_lowlight_score(img_tensor)
        compression_raw = self.compute_compression_score(img_tensor)
        
        # Independent normalization to [0,1]
        eps = 1e-6
        laplacian_var = -blur_raw
        blur_sev = 1.0 / (laplacian_var + eps)
        blur_sev = np.clip(np.log1p(blur_sev * 10) / 5.0, 0.0, 1.0)
        noise_sev = np.clip(np.log1p(noise_raw * 100) / 3.0, 0.0, 1.0)
        lowlight_sev = np.clip(lowlight_raw, 0.0, 1.0)
        compression_sev = np.clip(np.log1p(compression_raw * 10) / 2.0, 0.0, 1.0)
        
        total_score = (
            self.weights["blur"] * blur_sev +
            self.weights["noise"] * noise_sev +
            self.weights["lowlight"] * lowlight_sev +
            self.weights["compression"] * compression_sev
        )
        return float(np.clip(total_score, 0.0, 1.0))
    
    def sample_patch(self, input_img: Image.Image, target_img: Image.Image) -> Tuple[Image.Image, Image.Image]:
        w, h = input_img.size
        ps = self.patch_size
        if w < ps or h < ps:
            scale = max(ps / w, ps / h)
            new_w, new_h = int(w * scale), int(h * scale)
            input_img = input_img.resize((new_w, new_h), Image.BILINEAR)
            target_img = target_img.resize((new_w, new_h), Image.BILINEAR)
            w, h = new_w, new_h
        if self.mode == "train" and self.num_candidates > 1:
            candidates = []
            for _ in range(self.num_candidates * 2):
                left = random.randint(0, max(0, w - ps))
                top = random.randint(0, max(0, h - ps))
                patch_input = TF.crop(input_img, top, left, ps, ps)
                patch_tensor = TF.to_tensor(patch_input)
                edge_density = self.compute_edge_density(patch_tensor)
                if edge_density < self.min_edge_density:
                    continue
                degradation_score = self.compute_degradation_score(patch_tensor)
                candidates.append((left, top, degradation_score))
                if len(candidates) >= self.num_candidates:
                    break
            if candidates:
                candidates.sort(key=lambda x: x[2], reverse=True)
                left, top, _ = candidates[0]
            else:
                left = random.randint(0, max(0, w - ps))
                top = random.randint(0, max(0, h - ps))
        else:
            left = random.randint(0, max(0, w - ps)) if w > ps else 0
            top = random.randint(0, max(0, h - ps)) if h > ps else 0
        input_patch = TF.crop(input_img, top, left, ps, ps)
        target_patch = TF.crop(target_img, top, left, ps, ps)
        return input_patch, target_patch


class UnifiedRestorationDataset(Dataset):
    """Dataset that handles multiple datasets via CSV index with degradation-aware patch sampling"""
    
    def __init__(
        self,
        index_csv: str,
        patch_size: int = 256,
        mode: str = "train",
        normalize: Optional[Dict] = None,
        adaptive_sampling: bool = True,
        num_candidates: int = 5,
        handle_small_images: str = "resize"
    ):
        self.index_csv = Path(index_csv)
        self.patch_size = patch_size
        self.mode = mode
        self.normalize = normalize
        self.adaptive_sampling = adaptive_sampling and mode == "train"
        self.handle_small_images = handle_small_images
        self.pairs = self._load_index()
        
        if self.adaptive_sampling:
            dataset_name = self.pairs[0].get('dataset', 'GoPro') if self.pairs else 'GoPro'
            self.patch_sampler = DegradationAwarePatchSampler(patch_size, num_candidates, mode, dataset_name)
        else:
            dataset_name = self.pairs[0].get('dataset', 'GoPro') if self.pairs else 'GoPro'
            self.patch_sampler = DegradationAwarePatchSampler(patch_size, 1, mode, dataset_name)
    
    def _load_index(self) -> List[Dict]:
        pairs = []
        if not self.index_csv.exists():
            print(f"Warning: Index file not found: {self.index_csv}")
            return pairs
        with open(self.index_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                pairs.append(row)
        return pairs
    
    def _load_image(self, path: str) -> Image.Image:
        try:
            img = Image.open(path)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            return img
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            return Image.new('RGB', (self.patch_size, self.patch_size), (0, 0, 0))
    
    def _handle_small_image(self, img: Image.Image) -> Image.Image:
        """Handle images smaller than patch size"""
        # If patch_size is None (validation mode), return image as-is
        if self.patch_size is None:
            return img
        
        w, h = img.size
        ps = self.patch_size
        if w >= ps and h >= ps:
            return img
        if self.handle_small_images == "resize":
            scale = max(ps / w, ps / h)
            new_w, new_h = int(w * scale), int(h * scale)
            return img.resize((new_w, new_h), Image.BILINEAR)
        else:
            new_img = Image.new('RGB', (ps, ps), (0, 0, 0))
            left = (ps - w) // 2
            top = (ps - h) // 2
            new_img.paste(img, (left, top))
            return new_img
    
    def _augment_pair(self, input_img: Image.Image, target_img: Image.Image) -> Tuple[Image.Image, Image.Image]:
        if self.mode != "train":
            return input_img, target_img
        if random.random() < 0.5:
            input_img = TF.hflip(input_img)
            target_img = TF.hflip(target_img)
        if random.random() < 0.5:
            input_img = TF.vflip(input_img)
            target_img = TF.vflip(target_img)
        angle = random.choice([0, 90, 180, 270])
        if angle != 0:
            input_img = TF.rotate(input_img, angle)
            target_img = TF.rotate(target_img, angle)
        return input_img, target_img
    
    def _to_tensor_normalize(self, img: Image.Image) -> torch.Tensor:
        tensor = TF.to_tensor(img)
        if self.normalize:
            mean = torch.tensor(self.normalize['mean']).view(3, 1, 1)
            std = torch.tensor(self.normalize['std']).view(3, 1, 1)
            tensor = (tensor - mean) / std
        return tensor
    
    def __len__(self) -> int:
        return len(self.pairs)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        pair = self.pairs[idx]
        input_img = self._load_image(pair['input_path'])
        target_img = self._load_image(pair['target_path'])
        input_img = self._handle_small_image(input_img)
        target_img = self._handle_small_image(target_img)
        if self.patch_size:
            input_img, target_img = self.patch_sampler.sample_patch(input_img, target_img)
        input_img, target_img = self._augment_pair(input_img, target_img)
        input_tensor = self._to_tensor_normalize(input_img)
        target_tensor = self._to_tensor_normalize(target_img)
        return {
            'input': input_tensor,
            'target': target_tensor,
            'dataset': pair.get('dataset', 'unknown'),
            'scene': pair.get('scene', 'unknown')
        }


def create_dataloader(
    index_csv: str,
    batch_size: int = 8,
    patch_size: int = 256,
    mode: str = "train",
    normalize: Optional[Dict] = None,
    num_workers: int = 4,
    adaptive_sampling: bool = True,
    num_candidates: int = 5
) -> DataLoader:
    """Create a single-dataset dataloader"""
    dataset = UnifiedRestorationDataset(
        index_csv=index_csv,
        patch_size=patch_size,
        mode=mode,
        normalize=normalize,
        adaptive_sampling=adaptive_sampling,
        num_candidates=num_candidates
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(mode == "train"),
        num_workers=num_workers,
        pin_memory=True,
        drop_last=(mode == "train")
    )
    return loader


print("✓ Data preprocessing components defined (self-contained)")


✓ Data preprocessing components defined (self-contained)


In [5]:
# ============================================================================
# LOSS FUNCTIONS AND METRICS
# ============================================================================

def charbonnier_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    """Charbonnier loss (smooth L1)."""
    diff = pred - target
    return torch.sqrt(diff * diff + eps * eps).mean()


def ssim_loss(pred: torch.Tensor, target: torch.Tensor, window_size: int = 11) -> torch.Tensor:
    """
    Simplified SSIM loss (negative SSIM).
    
    Note: This is a simplified version. For production, use pytorch-msssim.
    """
    # Simple implementation: use MSE in perceptual space
    pred_gray = 0.299 * pred[:, 0] + 0.587 * pred[:, 1] + 0.114 * pred[:, 2]
    target_gray = 0.299 * target[:, 0] + 0.587 * target[:, 1] + 0.114 * target[:, 2]
    
    # Simple structural similarity approximation
    mu_pred = pred_gray.mean(dim=[1, 2], keepdim=True)
    mu_target = target_gray.mean(dim=[1, 2], keepdim=True)
    
    sigma_pred = pred_gray.std(dim=[1, 2], keepdim=True)
    sigma_target = target_gray.std(dim=[1, 2], keepdim=True)
    
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim = ((2 * mu_pred * mu_target + c1) * (2 * sigma_pred * sigma_target + c2)) / \
           ((mu_pred ** 2 + mu_target ** 2 + c1) * (sigma_pred ** 2 + sigma_target ** 2 + c2))
    
    return (1 - ssim.mean()) / 2.0  # Normalize to [0, 1]


def compute_loss(pred: torch.Tensor, target: torch.Tensor, config: Dict) -> Tuple[torch.Tensor, Dict]:
    """
    Compute combined loss.
    
    Returns:
        total_loss: Combined loss
        loss_dict: Dictionary with individual loss components
    """
    loss_dict = {}
    
    # L1 or Charbonnier loss
    if config["use_charbonnier"]:
        l1_loss = charbonnier_loss(pred, target, config["charbonnier_eps"])
        loss_dict["charbonnier"] = l1_loss.item()
    else:
        l1_loss = F.l1_loss(pred, target)
        loss_dict["l1"] = l1_loss.item()
    
    total_loss = config["l1_weight"] * l1_loss
    
    # SSIM loss (optional)
    if config["ssim_weight"] > 0:
        ssim = ssim_loss(pred, target)
        loss_dict["ssim"] = ssim.item()
        total_loss = total_loss + config["ssim_weight"] * ssim
    
    loss_dict["total"] = total_loss.item()
    return total_loss, loss_dict


def compute_psnr(pred: torch.Tensor, target: torch.Tensor, max_val: float = 1.0) -> float:
    """Compute PSNR (Peak Signal-to-Noise Ratio)."""
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(max_val) - 10 * np.log10(mse)
    return psnr


def compute_ssim_metric(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Compute SSIM metric (0-1, higher is better)."""
    # Simplified SSIM computation
    pred_gray = 0.299 * pred[:, 0] + 0.587 * pred[:, 1] + 0.114 * pred[:, 2]
    target_gray = 0.299 * target[:, 0] + 0.587 * target[:, 1] + 0.114 * target[:, 2]
    
    mu_pred = pred_gray.mean().item()
    mu_target = target_gray.mean().item()
    
    sigma_pred = pred_gray.std().item()
    sigma_target = target_gray.std().item()
    
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim = ((2 * mu_pred * mu_target + c1) * (2 * sigma_pred * sigma_target + c2)) / \
           ((mu_pred ** 2 + mu_target ** 2 + c1) * (sigma_pred ** 2 + sigma_target ** 2 + c2))
    
    return float(ssim)


print("✓ Loss functions and metrics defined")

✓ Loss functions and metrics defined


In [6]:
# ============================================================================
# CREATE DATALOADERS
# ============================================================================
# Uses UnifiedRestorationDataset from 01_data_preprocessing.ipynb
# ============================================================================

def create_train_dataloader(sanity_mode: bool = False) -> DataLoader:
    """Create training dataloader."""
    loader = create_dataloader(
        index_csv=DATASET_CONFIG["train_csv"],
        batch_size=TRAIN_CONFIG["batch_size"],
        patch_size=TRAIN_CONFIG["patch_size"],
        mode="train",
        adaptive_sampling=DATASET_CONFIG["adaptive_sampling"],
        num_candidates=DATASET_CONFIG["num_candidates"],
        num_workers=TRAIN_CONFIG["num_workers"]
    )
    
    # For sanity test: limit to small subset
    if sanity_mode:
        # Create a subset dataset
        full_dataset = loader.dataset
        indices = list(range(min(SANITY_CONFIG["num_samples"], len(full_dataset))))
        from torch.utils.data import Subset
        subset = Subset(full_dataset, indices)
        loader = DataLoader(
            subset,
            batch_size=TRAIN_CONFIG["batch_size"],
            shuffle=True,
            num_workers=TRAIN_CONFIG["num_workers"],
            pin_memory=True
        )
        print(f"  Sanity mode: Using {len(subset)} samples")
    
    return loader


def create_val_dataloader() -> DataLoader:
    """Create validation dataloader (limited size for faster validation)."""
    loader = create_dataloader(
        index_csv=DATASET_CONFIG["val_csv"],
        batch_size=1,  # Batch size 1 for validation (full images)
        patch_size=None,  # No patching for validation
        mode="test",
        adaptive_sampling=False,
        num_workers=TRAIN_CONFIG["num_workers"]
    )
    
    # Limit validation set size for faster validation
    max_samples = VAL_CONFIG.get("max_val_samples", None)
    if max_samples and len(loader.dataset) > max_samples:
        from torch.utils.data import Subset
        indices = list(range(min(max_samples, len(loader.dataset))))
        subset = Subset(loader.dataset, indices)
        loader = DataLoader(
            subset,
            batch_size=1,
            shuffle=False,
            num_workers=TRAIN_CONFIG["num_workers"],
            pin_memory=True
        )
        print(f"  Validation set limited to {len(subset)} samples (from {len(loader.dataset)})")
    
    return loader


# Create dataloaders
print("Creating dataloaders...")
train_loader = create_train_dataloader(sanity_mode=False)
val_loader = create_val_dataloader()

print(f"\n{'='*60}")
print(f"Data Loaders")
print(f"{'='*60}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"Patch size: {TRAIN_CONFIG['patch_size']}")
print(f"{'='*60}\n")

Creating dataloaders...
  Validation set limited to 100 samples (from 100)

Data Loaders
Train batches: 262
Val batches: 100
Batch size: 8
Patch size: 256



In [7]:
# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================

def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    scaler: Optional[GradScaler],
    config: Dict,
    loss_config: Dict,
    device: torch.device
) -> Dict:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    loss_components = defaultdict(float)
    num_batches = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        input_img = batch['input'].to(device)
        target_img = batch['target'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass with AMP if enabled
        if config["use_amp"] and device.type == "cuda":
            with autocast():
                output = model(input_img)
                loss, loss_dict = compute_loss(output, target_img, loss_config)
            
            scaler.scale(loss).backward()
            
            # Gradient clipping
            if config["grad_clip"] > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            
            scaler.step(optimizer)
            scaler.update()
        else:
            output = model(input_img)
            loss, loss_dict = compute_loss(output, target_img, loss_config)
            loss.backward()
            
            # Gradient clipping
            if config["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            
            optimizer.step()
        
        # Accumulate losses
        total_loss += loss.item()
        for key, value in loss_dict.items():
            loss_components[key] += value
        num_batches += 1
        
        # Update progress bar
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    # Average losses
    avg_loss = total_loss / num_batches
    avg_components = {k: v / num_batches for k, v in loss_components.items()}
    
    return {"loss": avg_loss, **avg_components}


def validate(
    model: nn.Module,
    loader: DataLoader,
    loss_config: Dict,
    device: torch.device,
    save_images: bool = False,
    num_images: int = 4,
    epoch: int = 0,
    output_dir: Optional[Path] = None
) -> Dict:
    """Validate model."""
    model.eval()
    
    total_loss = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    num_batches = 0
    
    saved_images = []
    
    with torch.no_grad():
        for idx, batch in enumerate(tqdm(loader, desc="Validating")):
            input_img = batch['input'].to(device)
            target_img = batch['target'].to(device)
            
            # Forward pass
            output = model(input_img)
            
            # Compute metrics
            loss, _ = compute_loss(output, target_img, loss_config)
            psnr = compute_psnr(output, target_img)
            ssim = compute_ssim_metric(output, target_img)
            
            total_loss += loss.item()
            total_psnr += psnr
            total_ssim += ssim
            num_batches += 1
            
            # Save images for visualization
            if save_images and len(saved_images) < num_images:
                saved_images.append({
                    'input': input_img[0].cpu(),
                    'output': output[0].cpu(),
                    'target': target_img[0].cpu()
                })
    
    # Average metrics
    avg_loss = total_loss / num_batches
    avg_psnr = total_psnr / num_batches
    avg_ssim = total_ssim / num_batches
    
    # Save image grid
    if save_images and saved_images and output_dir:
        save_image_grid(saved_images, epoch, output_dir)
    
    return {
        "loss": avg_loss,
        "psnr": avg_psnr,
        "ssim": avg_ssim
    }


def save_image_grid(images: List[Dict], epoch: int, output_dir: Path):
    """Save a grid of input/output/target images."""
    num_images = len(images)
    fig, axes = plt.subplots(num_images, 3, figsize=(12, 4 * num_images))
    
    if num_images == 1:
        axes = axes.reshape(1, -1)
    
    for i, img_dict in enumerate(images):
        input_img = img_dict['input'].permute(1, 2, 0).cpu().numpy()
        output_img = img_dict['output'].permute(1, 2, 0).cpu().numpy()
        target_img = img_dict['target'].permute(1, 2, 0).cpu().numpy()
        
        # Clamp to [0, 1]
        input_img = np.clip(input_img, 0, 1)
        output_img = np.clip(output_img, 0, 1)
        target_img = np.clip(target_img, 0, 1)
        
        axes[i, 0].imshow(input_img)
        axes[i, 0].set_title(f"Input (Epoch {epoch})")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(output_img)
        axes[i, 1].set_title(f"Output (Epoch {epoch})")
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(target_img)
        axes[i, 2].set_title(f"Target (Epoch {epoch})")
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_dir / f"val_images_epoch_{epoch:03d}.png", dpi=150, bbox_inches='tight')
    plt.close()


print("✓ Training functions defined")

✓ Training functions defined


In [8]:
# ============================================================================
# METRICS LOGGING
# ============================================================================

class MetricsLogger:
    """Simple CSV logger for training metrics."""
    def __init__(self, csv_path: str):
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self.metrics = []
        
    def log(self, epoch: int, train_metrics: Dict, val_metrics: Dict, time_elapsed: float):
        """Log metrics for an epoch."""
        row = {
            'epoch': epoch,
            'train_loss': train_metrics.get('loss', 0.0),
            'val_loss': val_metrics.get('loss', 0.0),
            'val_psnr': val_metrics.get('psnr', 0.0),
            'val_ssim': val_metrics.get('ssim', 0.0),
            'time_seconds': time_elapsed
        }
        self.metrics.append(row)
        
        # Write to CSV
        if self.csv_path.exists():
            mode = 'a'
        else:
            mode = 'w'
        
        with open(self.csv_path, mode, newline='') as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            if mode == 'w':
                writer.writeheader()
            writer.writerow(row)
    
    def print_summary(self, epoch: int, train_metrics: Dict, val_metrics: Dict):
        """Print formatted metrics summary."""
        print(f"\nEpoch {epoch:3d} | "
              f"Train Loss: {train_metrics['loss']:.4f} | "
              f"Val Loss: {val_metrics['loss']:.4f} | "
              f"PSNR: {val_metrics['psnr']:.2f} dB | "
              f"SSIM: {val_metrics['ssim']:.4f}")


logger = MetricsLogger(VAL_CONFIG["metrics_csv"])
print(f"✓ Metrics logger initialized: {VAL_CONFIG['metrics_csv']}")

✓ Metrics logger initialized: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear/experiments/baseline_training/metrics.csv


In [9]:
# ============================================================================
# SANITY OVERFIT TEST
# ============================================================================
# This tests if the training pipeline works correctly by overfitting on a tiny subset
# ============================================================================

def run_sanity_test() -> bool:
    """
    Run sanity overfit test.
    
    Returns:
        True if sanity test passes, False otherwise
    """
    print(f"\n{'='*60}")
    print(f"SANITY OVERFIT TEST")
    print(f"{'='*60}")
    print(f"Training on {SANITY_CONFIG['num_samples']} samples for {SANITY_CONFIG['epochs']} epochs")
    print(f"Expected: Loss should drop by at least {SANITY_CONFIG['expected_loss_drop']}")
    print(f"{'='*60}\n")
    
    # Create small dataloader
    sanity_loader = create_train_dataloader(sanity_mode=True)
    
    # Create fresh model
    sanity_model = BaselineRestorationModel(
        base_channels=MODEL_CONFIG["base_channels"],
        num_blocks=MODEL_CONFIG["num_blocks"],
        dropout=MODEL_CONFIG["dropout"]
    ).to(DEVICE)
    
    # Create optimizer
    sanity_optimizer = optim.AdamW(
        sanity_model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"]
    )
    
    # Create scaler for AMP
    sanity_scaler = GradScaler() if (TRAIN_CONFIG["use_amp"] and DEVICE.type == "cuda") else None
    
    # Track initial loss
    initial_loss = None
    final_loss = None
    
    # Train for few epochs
    for epoch in range(1, SANITY_CONFIG["epochs"] + 1):
        train_metrics = train_epoch(
            sanity_model, sanity_loader, sanity_optimizer, sanity_scaler,
            TRAIN_CONFIG, LOSS_CONFIG, DEVICE
        )
        
        if initial_loss is None:
            initial_loss = train_metrics['loss']
        final_loss = train_metrics['loss']
        
        print(f"Epoch {epoch}/{SANITY_CONFIG['epochs']} | Loss: {train_metrics['loss']:.4f}")
    
    # Check if loss dropped sufficiently
    loss_drop = initial_loss - final_loss
    loss_drop_ratio = loss_drop / initial_loss if initial_loss > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"Sanity Test Results")
    print(f"{'='*60}")
    print(f"Initial loss: {initial_loss:.4f}")
    print(f"Final loss: {final_loss:.4f}")
    print(f"Loss drop: {loss_drop:.4f} ({loss_drop_ratio*100:.1f}%)")
    
    if loss_drop_ratio >= SANITY_CONFIG["expected_loss_drop"]:
        print(f"✓ SANITY TEST PASSED: Loss dropped sufficiently!")
        print(f"{'='*60}\n")
        return True
    else:
        print(f"✗ SANITY TEST FAILED: Loss did not drop enough.")
        print(f"\nDebugging hints:")
        print(f"  - Check if input/target images are properly paired")
        print(f"  - Verify image normalization (should be [0, 1])")
        print(f"  - Check if model forward pass works correctly")
        print(f"  - Verify loss computation")
        print(f"  - Check if optimizer is updating weights")
        print(f"{'='*60}\n")
        return False


# Run sanity test if enabled
if SANITY_CONFIG["enabled"]:
    sanity_passed = run_sanity_test()
    if not sanity_passed:
        print("⚠️  Sanity test failed. Please fix issues before proceeding.")
        print("    You can set SANITY_CONFIG['enabled'] = False to skip this test.")
else:
    print("Sanity test disabled. Set SANITY_CONFIG['enabled'] = True to enable.")


SANITY OVERFIT TEST
Training on 100 samples for 5 epochs
Expected: Loss should drop by at least 0.3

  Sanity mode: Using 100 samples


Training: 100%|████████████████████| 13/13 [00:43<00:00,  3.33s/it, loss=0.1146]


Epoch 1/5 | Loss: 0.2145


Training: 100%|████████████████████| 13/13 [00:41<00:00,  3.21s/it, loss=0.0770]


Epoch 2/5 | Loss: 0.0995


...

Epoch 3/5 | Loss: 0.0700


Training: 100%|████████████████████| 13/13 [00:42<00:00,  3.24s/it, loss=0.0552]


Epoch 4/5 | Loss: 0.0626


Training: 100%|████████████████████| 13/13 [00:41<00:00,  3.21s/it, loss=0.0730]

Epoch 5/5 | Loss: 0.0606

Sanity Test Results
Initial loss: 0.2145
Final loss: 0.0606
Loss drop: 0.1538 (71.7%)
✓ SANITY TEST PASSED: Loss dropped sufficiently!



In [10]:
# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================

# Create optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"]
)

# Create learning rate scheduler
if TRAIN_CONFIG["scheduler"] == "cosine":
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_CONFIG["scheduler_params"]["T_max"],
        eta_min=TRAIN_CONFIG["scheduler_params"]["eta_min"]
    )
elif TRAIN_CONFIG["scheduler"] == "step":
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=TRAIN_CONFIG["scheduler_params"]["step_size"],
        gamma=TRAIN_CONFIG["scheduler_params"]["gamma"]
    )
else:
    scheduler = None

# Create scaler for AMP
scaler = GradScaler() if (TRAIN_CONFIG["use_amp"] and DEVICE.type == "cuda") else None

# Resume training if enabled
start_epoch = 1
best_val_psnr = 0.0
best_epoch = 0

if RESUME_CONFIG["enabled"]:
    checkpoint_path = Path(RESUME_CONFIG["checkpoint_path"])
    if checkpoint_path.exists():
        print(f"\n{'='*60}")
        print(f"Resuming Training")
        print(f"{'='*60}")
        print(f"Loading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if scheduler and 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint.get('epoch', 1) + 1
        best_val_psnr = checkpoint.get('best_val_psnr', 0.0)
        best_epoch = checkpoint.get('best_epoch', 0)
        print(f"Resumed from epoch {start_epoch}")
        print(f"Best PSNR so far: {best_val_psnr:.2f} dB (epoch {best_epoch})")
        print(f"{'='*60}\n")
    else:
        print(f"⚠️  Warning: Checkpoint not found: {checkpoint_path}")
        print("   Starting training from scratch.")

# Training history
history = []

print(f"\n{'='*60}")
print(f"Starting Training")
print(f"{'='*60}")
print(f"Total epochs: {TRAIN_CONFIG['epochs']}")
print(f"Starting from epoch: {start_epoch}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"{'='*60}\n")

# Training loop
for epoch in range(start_epoch, TRAIN_CONFIG["epochs"] + 1):
    epoch_start_time = time.time()
    
    # Train
    train_metrics = train_epoch(
        model, train_loader, optimizer, scaler,
        TRAIN_CONFIG, LOSS_CONFIG, DEVICE
    )
    
    # Update learning rate
    if scheduler:
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
    else:
        current_lr = TRAIN_CONFIG["learning_rate"]
    
    # Validate
    if epoch % VAL_CONFIG["val_interval"] == 0:
        val_metrics = validate(
            model, val_loader, LOSS_CONFIG, DEVICE,
            save_images=VAL_CONFIG["save_images"],
            num_images=VAL_CONFIG["num_val_images"],
            epoch=epoch,
            output_dir=OUTPUT_DIR
        )
        
        # Log metrics
        time_elapsed = time.time() - epoch_start_time
        logger.log(epoch, train_metrics, val_metrics, time_elapsed)
        logger.print_summary(epoch, train_metrics, val_metrics)
        
        # Save best model
        if val_metrics["psnr"] > best_val_psnr:
            best_val_psnr = val_metrics["psnr"]
            best_epoch = epoch
            
            # Save checkpoint
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'scaler_state_dict': scaler.state_dict() if scaler else None,
                'best_val_psnr': best_val_psnr,
                'best_epoch': best_epoch,
                'val_psnr': val_metrics['psnr'],
                'val_ssim': val_metrics['ssim'],
                'config': {
                    'model': MODEL_CONFIG,
                    'train': TRAIN_CONFIG,
                    'loss': LOSS_CONFIG
                }
            }
            # Save to checkpoints directory
            checkpoint_dir = OUTPUT_DIR / "checkpoints"
            checkpoint_dir.mkdir(exist_ok=True)
            torch.save(checkpoint, checkpoint_dir / "best_model.pth")
            torch.save(checkpoint, OUTPUT_DIR / "best_model.pth")
            print(f"  → Saved best model (PSNR: {best_val_psnr:.2f} dB)")
        
        history.append({
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'val_loss': val_metrics['loss'],
            'val_psnr': val_metrics['psnr'],
            'val_ssim': val_metrics['ssim']
        })
    
    # Save checkpoint periodically
    if epoch % VAL_CONFIG["save_interval"] == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'best_val_psnr': best_val_psnr,
            'best_epoch': best_epoch,
            'val_psnr': val_metrics.get('psnr', 0.0) if epoch % VAL_CONFIG["val_interval"] == 0 else 0.0,
            'val_ssim': val_metrics.get('ssim', 0.0) if epoch % VAL_CONFIG["val_interval"] == 0 else 0.0
        }
        checkpoint_dir = OUTPUT_DIR / "checkpoints"
        checkpoint_dir.mkdir(exist_ok=True)
        torch.save(checkpoint, checkpoint_dir / f"checkpoint_epoch_{epoch:03d}.pth")

# Save final checkpoint
final_checkpoint = {
    'epoch': TRAIN_CONFIG['epochs'],
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
    'val_psnr': best_val_psnr,
    'val_ssim': history[-1]['val_ssim'] if history else 0.0
}
torch.save(final_checkpoint, OUTPUT_DIR / "last_model.pth")                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"{'='*60}")
print(f"Best validation PSNR: {best_val_psnr:.2f} dB (epoch {best_epoch})")
print(f"Checkpoints saved to: {OUTPUT_DIR}")
print(f"Metrics CSV: {VAL_CONFIG['metrics_csv']}")
print(f"{'='*60}\n")


Starting Training
Total epochs: 5
Starting from epoch: 1
Train batches per epoch: 262
Validation batches: 100



Validating: 100%|█████████████████████████████| 100/100 [02:30<00:00,  1.51s/it]



Epoch   5 | Train Loss: 0.0401 | Val Loss: 0.0227 | PSNR: 27.87 dB | SSIM: 0.9990
  → Saved best model (PSNR: 27.87 dB)

Training Complete!
Best validation PSNR: 27.87 dB (epoch 5)
Checkpoints saved to: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear/experiments/baseline_training
Metrics CSV: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear/experiments/baseline_training/metrics.csv

